# Part 2: ResNet50 Model Improvement Ablations

This notebook keeps the Part 2 workflow notebook-owned while moving reusable training and result logic into project modules. The regular-training baseline is loaded from Part 1 results; the remaining ablations are trained here.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
import pandas as pd
from IPython.display import Image, display

from src.evaluation.experiment_results import (
    experiment_output_paths,
    get_device,
    load_part1_model_baseline_aggregated,
)
from src.training.experiment_steps import run_part2_improvement_experiments
from src.utils.config import Part2ExperimentConfig
from src.utils.reproducibility import seed_everything

## Configuration

In [ ]:
config = Part2ExperimentConfig()
device = get_device(config)
output_paths = experiment_output_paths(config.results_dir, config.figures_dir, config.part)
seed_everything(config.seed, deterministic=config.deterministic)

pd.DataFrame(
    [
        {
            'part': config.part,
            'config_name': config.config_name,
            'model_name': config.model_name,
            'grid_sizes': config.grid_sizes,
            'num_permutations': config.num_permutations,
            'epochs': config.epochs,
            'device': str(device),
        }
    ]
)

In [ ]:
pd.DataFrame(config.ablations)

## Part 1 ResNet50 Baseline

In [ ]:
part1_baseline_aggregated = load_part1_model_baseline_aggregated(config, config.model_name)
if not part1_baseline_aggregated.empty:
    display(part1_baseline_aggregated)

## Train Improvement Ablations

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    part2_results = run_part2_improvement_experiments(config=config, device=device)
    display(part2_results)
else:
    print('Training is skipped. Set RUN_TRAINING = True to train Part 2 ablations in this notebook.')

## Results

In [ ]:
results_path = Path(output_paths['aggregated_results'])

if results_path.exists():
    part2_results = pd.read_csv(results_path)
    if 'regular_part1' not in set(part2_results.get('ablation_name', [])) and not part1_baseline_aggregated.empty:
        part2_results = pd.concat([part1_baseline_aggregated, part2_results], ignore_index=True, sort=False)
    display(part2_results.sort_values(['grid_size', 'ablation_name']))
else:
    print('Part 2 aggregated results were not found. Set RUN_TRAINING = True and run the training cell.')
    if part1_baseline_aggregated.empty:
        print('Part 1 ResNet50 baseline results were also not found; run Part 1 first to include regular_part1.')

In [ ]:
figure_path = Path(output_paths['figure'])
if figure_path.exists():
    display(Image(filename=str(figure_path)))
else:
    print('Part 2 ablation figure has not been generated yet.')